# Ablation: LR regularization vs. class weighting (E5 Hybrid, C = 0.1, unweighted)

Runs E1 (needed as reference) plus **E5 Hybrid with C = 0.1 and no class weights**, which separates the effect of class weighting from the stronger LR regularization. Same environment as the main notebook; about 40 min on a Kaggle GPU session (CPU work).

Inputs: SemEval-2025 Task 11, PRDECT-ID, and (recommended) the embedding-cache dataset.

Outputs correspond to the rows marked ‡ in Table 8 of the manuscript (`table_ablationC.csv`). Because this run uses a separate Kaggle session, its XGBoost-based scores differ slightly from the main run; contrasts are therefore computed within this run (reference models re-run here).

> **Label note.** `ME-XBAL` was a working name used during the experiments and is kept only as an internal label in the code and result files so that they stay identical to the reported run. In the manuscript, `ME-XBAL` = **E5 Hybrid (imbalance-aware)** and `ME-XBAL (XLM-R)` = **XLM-R Hybrid (imbalance-aware)**.


In [ ]:
# ============================================================
# 0. INSTALL & IMPORTS
# ============================================================
!pip -q install iterative-stratification shap

import os, sys, re, time, json, glob, random, warnings, gc, platform, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy import stats
import joblib

import sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
import xgboost as xgb
import transformers
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit, MultilabelStratifiedKFold

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| sklearn", sklearn.__version__, "| xgboost", xgb.__version__)
print("Device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Configuration

In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================
SMOKE_TEST = bool(int(os.environ.get("MEXBAL_SMOKE", "0")))   # keep 0 on Kaggle (full run)

SEEDS     = [42, 7, 13, 77, 101, 202, 303, 404, 505, 606]
LC_SEEDS  = [42, 7, 13, 77, 101]
SEED_PRIMARY = 42
LC_SIZES  = [100, 250, 500, 750, 1000, 1500, 2000, 2500, 3000, 3510]
MATCHED_SIZES = LC_SIZES                 # E2b sizes (equal data budget)

VAL_FRAC, TEST_FRAC = 0.15, 0.20
N_OOF_FOLDS = 5
BOOTSTRAP_REPS = 3000

# ---- switches ------------------------------------------------
RUN_MAIN          = True    # E1/E3 (+ ME-XBAL and ME-XBAL (XLM-R))
RUN_LEARNING_CURVE= False    # E2 ME-XBAL curve + E2b
RUN_FT_E5         = False    # E2 fine-tuned multilingual-E5 (encoder-matched comparator)
RUN_FT_XLMR       = False   # E2 fine-tuned XLM-R (optional, ~3 h extra GPU time)
RUN_SHAP          = False

# ---- encoders ----------------------------------------------------
ENCODERS = {
    "e5":       dict(name="intfloat/multilingual-e5-base", prefix="query: "),
    "xlmr":     dict(name="xlm-roberta-base",              prefix=None),
    "indobert": dict(name="indobenchmark/indobert-base-p1", prefix=None),
}
MAIN_ENCODER = "e5"            # encoder of ME-XBAL
MAX_LEN, EMBED_BATCH = 128, 16

# ---- fine-tuning (E2) ---------------------------------------
FT_BATCH_SIZE, FT_LR, FT_WEIGHT_DECAY = 16, 2e-5, 0.01
LC_FT_MAX_EPOCHS, LC_FT_PATIENCE = 5, 2

# ---- selection grids (identical for all models) ----------------
WEIGHT_GRID    = np.arange(0.0, 1.0001, 0.05)
THRESHOLD_GRID = np.arange(0.15, 0.8501, 0.01)
MIN_PRECISION, MAX_PPR_RATIO, TIE_TOL = 0.20, 3.0, 0.005
ALPHA_GRID, ALPHA_PASSES = np.arange(0.50, 3.0001, 0.05), 3

XGB_COMMON = dict(max_depth=3, min_child_weight=5, learning_rate=0.05, n_estimators=200,
                  subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.5,
                  verbosity=0, n_jobs=-1)
XGB_PARAMS_BINARY = dict(objective="binary:logistic", eval_metric="logloss", **XGB_COMMON)
XGB_PARAMS_MULTI  = dict(objective="multi:softprob", eval_metric="mlogloss", **XGB_COMMON)

# ---- SHAP ----------------------------------------------------
SHAP_MAX_EVALS, SHAP_BATCH_SIZE = 300, 8
SHAP_TOK_N, SHAP_SEM_PER_LANG = 30, 10
SHAP_FRACTIONS, SHAP_RANDOM_REPS = [0.10, 0.25, 0.50], 5

if SMOKE_TEST:   # tiny configuration used only to check that every cell runs
    SEEDS, LC_SEEDS = [42, 7, 13], [42, 7]
    LC_SIZES = MATCHED_SIZES = [100, 500]
    LC_FT_MAX_EPOCHS, BOOTSTRAP_REPS = 1, 200
    SHAP_TOK_N, SHAP_SEM_PER_LANG, SHAP_MAX_EVALS = 10, 2, 60

# ---- paths ------------------------------------------------------
def _find(pattern, roots=("/kaggle/input", os.environ.get("MEXBAL_DATA", "/nonexistent"))):
    for r in roots:
        hits = sorted(glob.glob(os.path.join(r, "**", pattern), recursive=True))
        if hits: return hits[0]
    return None

def _semeval_root():
    # the SemEval release contains several folders; use the one whose dev files carry gold labels
    roots = ("/kaggle/input", os.environ.get("MEXBAL_DATA", "/nonexistent"))
    hits = [h for r in roots for h in sorted(glob.glob(os.path.join(r, "**", "track_a/train/sun.csv"), recursive=True))]
    hits.sort(key=lambda h: "public_data_test" not in h)          # the folder used in the paper first
    for h in hits:
        root = Path(h).parents[2]
        need = [root / "track_a/dev/sun.csv", root / "track_c/dev/jav.csv", root / "track_c/dev/ind.csv"]
        if all(f.exists() for f in need) and all({"anger", "joy"} <= set(pd.read_csv(f, nrows=1).columns) for f in need):
            return str(root)
    return None

SEMEVAL_ROOT = _semeval_root()
assert SEMEVAL_ROOT, "SemEval-2025 Task 11 data (with labelled dev files) not found: add the dataset as Kaggle input."
TOKOPEDIA_CSV = _find("PRDECT-ID Dataset.csv") or _find("*PRDECT*.csv") or _find("Tokopedia Review.csv")
assert TOKOPEDIA_CSV, "PRDECT-ID (Tokopedia) CSV not found: add the dataset as Kaggle input."

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("MEXBAL_WORK", "./work")
OUT_ROOT  = os.path.join(WORK, "ME-XBAL_E5")
CACHE_OUT = os.path.join(WORK, "embedding_cache")
_cache_hit = _find("tokopedia_e5.npz")
CACHE_IN  = str(Path(_cache_hit).parent) if _cache_hit else None
for d in [OUT_ROOT, CACHE_OUT, f"{OUT_ROOT}/tables", f"{OUT_ROOT}/predictions", f"{OUT_ROOT}/SHAP"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("SemEval root :", SEMEVAL_ROOT)
print("Tokopedia csv:", TOKOPEDIA_CSV)
print("Cache input  :", CACHE_IN)
print("Output       :", OUT_ROOT, "| SMOKE_TEST =", SMOKE_TEST)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def append_csv(path, rows):
    # Append rows (list of dicts) to a CSV and return the full DataFrame
    new = pd.DataFrame(rows)
    old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    df = pd.concat([old, new], ignore_index=True)
    df.to_csv(path, index=False); return df

def done_keys(path, cols):
    if not os.path.exists(path): return set()
    df = pd.read_csv(path)
    return set(map(tuple, df[cols].astype(str).values.tolist()))

## 2. Utilities and statistics

In [ ]:
# ============================================================
# 2. UTILITIES
# ============================================================
def masked_mean_pool(h, mask):
    m = mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

def fit_preprocessor(X, seed=42):
    sc = StandardScaler().fit(X)
    pc = PCA(n_components=0.95, random_state=seed).fit(sc.transform(X))
    return (lambda A: pc.transform(sc.transform(A))), sc, pc

def corrected_ttest(d, ratio):
    # Nadeau-Bengio corrected resampled t-test on per-seed differences d
    d = np.asarray(d, float); J = len(d); m = d.mean()
    se = np.sqrt((1.0 / J + ratio) * d.var(ddof=1))
    if se == 0: return m, m, m, 1.0
    t = m / se; p = 2 * stats.t.sf(abs(t), J - 1); h = stats.t.ppf(0.975, J - 1) * se
    return m, m - h, m + h, p

def wilcoxon_p(a, b):
    d = np.asarray(a) - np.asarray(b)
    return float(stats.wilcoxon(a, b).pvalue) if np.any(d != 0) else 1.0

def holm(pvals):
    p = np.asarray(pvals, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for r, i in enumerate(o):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
    return adj

def bootstrap_mean_ci(v, reps=None, seed=42):
    reps = reps or BOOTSTRAP_REPS
    v = np.asarray(v, float); rng = np.random.default_rng(seed)
    means = v[rng.integers(0, len(v), size=(reps, len(v)))].mean(1)
    return float(v.mean()), *map(float, np.quantile(means, [0.025, 0.975]))

def paired_prediction_bootstrap(y, pa, pb, reps=None, seed=42):
    reps = reps or BOOTSTRAP_REPS
    y, pa, pb = map(np.asarray, (y, pa, pb)); rng = np.random.default_rng(seed); n = len(y)
    f = lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0)
    d = np.array([f(y[i], pa[i]) - f(y[i], pb[i]) for i in (rng.integers(0, n, n) for _ in range(reps))])
    return dict(difference=f(y, pa) - f(y, pb), ci95_low=float(np.quantile(d, .025)), ci95_high=float(np.quantile(d, .975)))

## 3. Data and fixed splits

In [ ]:
# ============================================================
# 3. DATA
# ============================================================
SEMEVAL_LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

def load_semeval():
    sun = pd.concat([pd.read_csv(f"{SEMEVAL_ROOT}/track_a/train/sun.csv"),
                     pd.read_csv(f"{SEMEVAL_ROOT}/track_a/dev/sun.csv")], ignore_index=True)
    jav = pd.read_csv(f"{SEMEVAL_ROOT}/track_c/dev/jav.csv")
    ind = pd.read_csv(f"{SEMEVAL_ROOT}/track_c/dev/ind.csv")
    data = {"sun": sun.reset_index(drop=True), "jav": jav.reset_index(drop=True), "ind": ind.reset_index(drop=True)}
    for lang, df in data.items():
        assert "text" in df.columns and all(c in df.columns for c in SEMEVAL_LABELS), lang
        df["uid"] = [f"{lang}_{i:05d}" for i in range(len(df))]
    return data

def load_tokopedia():
    df = pd.read_csv(TOKOPEDIA_CSV)[["Customer Review", "Emotion"]].dropna().reset_index(drop=True)
    le = LabelEncoder(); df["label"] = le.fit_transform(df["Emotion"])
    df["uid"] = [f"tok_{i:05d}" for i in range(len(df))]
    return df, list(le.classes_)

SEMEVAL_DATA = load_semeval()
TOKO_DF, TOKO_CLASSES = load_tokopedia()

def _two_stage(splitter, y, seed):
    hold = VAL_FRAC + TEST_FRAC
    tr, h = next(splitter(n_splits=1, test_size=hold, random_state=seed).split(np.zeros(len(y)), y))
    va, te = next(splitter(n_splits=1, test_size=TEST_FRAC / hold, random_state=seed).split(np.zeros(len(h)), y[h]))
    return np.sort(tr), np.sort(h[va]), np.sort(h[te])

ALL_SEEDS = sorted(set(SEEDS) | set(LC_SEEDS) | {SEED_PRIMARY}, key=lambda s: ([42,7,13,77,101,202,303,404,505,606]+[s]).index(s))
SEMEVAL_SPLITS = {(l, s): _two_stage(MultilabelStratifiedShuffleSplit, SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int), s)
                  for l in SEMEVAL_DATA for s in ALL_SEEDS}
TOKO_SPLITS = {s: _two_stage(StratifiedShuffleSplit, TOKO_DF["label"].values, s) for s in ALL_SEEDS}

for (l, s), (tr, va, te) in SEMEVAL_SPLITS.items():
    assert not (set(tr) & set(va) or set(tr) & set(te) or set(va) & set(te))
for s, (tr, va, te) in TOKO_SPLITS.items():
    assert not (set(tr) & set(va) or set(tr) & set(te) or set(va) & set(te))

manifest = []
for (l, s), parts in SEMEVAL_SPLITS.items():
    for sp, idx in zip(["train", "val", "test"], parts):
        manifest += [{"dataset": "SemEval", "lang": l, "seed": s, "split": sp, "uid": SEMEVAL_DATA[l]["uid"].values[i]} for i in idx]
for s, parts in TOKO_SPLITS.items():
    for sp, idx in zip(["train", "val", "test"], parts):
        manifest += [{"dataset": "Tokopedia", "lang": "ind", "seed": s, "split": sp, "uid": TOKO_DF["uid"].values[i]} for i in idx]
pd.DataFrame(manifest).to_csv(f"{OUT_ROOT}/split_manifest_all.csv", index=False)
print("SemEval:", {l: len(d) for l, d in SEMEVAL_DATA.items()}, "| Tokopedia:", len(TOKO_DF), TOKO_CLASSES)
print("Test sizes seed 42:", {l: len(SEMEVAL_SPLITS[(l, 42)][2]) for l in SEMEVAL_DATA}, len(TOKO_SPLITS[42][2]))

## 4. Frozen-encoder embeddings (cached)

In [ ]:
# ============================================================
# 4. EMBEDDINGS
# ============================================================
@torch.no_grad()
def encode(texts, model, tok, prefix=None, batch=EMBED_BATCH):
    texts = [(prefix or "") + str(x) for x in texts]
    rows = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i + batch], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        z = masked_mean_pool(model(**enc).last_hidden_state, enc["attention_mask"])
        rows.append(F.normalize(z, p=2, dim=1).cpu().numpy())
    return np.vstack(rows)

def cached_embeddings(cache_name, texts, enc_key):
    for root in [CACHE_IN, CACHE_OUT]:
        if root and os.path.exists(f"{root}/{cache_name}.npz"):
            emb = np.load(f"{root}/{cache_name}.npz")["embedding"]
            if emb.shape[0] == len(texts):
                print("[cache]", cache_name, emb.shape); return emb
    cfg = ENCODERS[enc_key]
    tok = AutoTokenizer.from_pretrained(cfg["name"])
    model = AutoModel.from_pretrained(cfg["name"]).to(DEVICE).eval()
    emb = encode(texts, model, tok, cfg["prefix"])
    np.savez_compressed(f"{CACHE_OUT}/{cache_name}.npz", embedding=emb)
    print("[computed]", cache_name, emb.shape)
    del model, tok; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return emb

SEMEVAL_EMB = {k: {l: cached_embeddings(f"semeval_{k}_{l}", SEMEVAL_DATA[l]["text"].tolist(), k) for l in SEMEVAL_DATA}
               for k in ENCODERS}
TOKO_EMB = {k: cached_embeddings(f"tokopedia_{k}", TOKO_DF["Customer Review"].tolist(), k) for k in ENCODERS}

## 5. Classifier engines (identical out-of-fold search for every model)

In [ ]:
# ============================================================
# 5. MULTI-LABEL ENGINE (SemEval)
# ============================================================
class ConstantBinary:
    def __init__(self, p): self.p = float(p)
    def predict_proba(self, X): return np.c_[np.full(len(X), 1 - self.p), np.full(len(X), self.p)]

def train_ml(X, y, cfg, seed):
    xm = {} if cfg["use_xgb"] else None; lm = {} if cfg["use_lr"] else None
    for j, e in enumerate(SEMEVAL_LABELS):
        yy = y[:, j]; pos = int(yy.sum()); neg = len(yy) - pos
        if cfg["use_xgb"]:
            if pos == 0 or neg == 0: xm[e] = ConstantBinary(yy.mean())
            else:
                ratio = neg / pos if cfg["cost_sensitive"] else 1.0
                xm[e] = xgb.XGBClassifier(scale_pos_weight=ratio, random_state=seed, **XGB_PARAMS_BINARY).fit(X, yy)
        if cfg["use_lr"]:
            if pos == 0 or neg == 0: lm[e] = ConstantBinary(yy.mean())
            else:
                lm[e] = LogisticRegression(C=cfg["lr_C"], max_iter=1000, solver="lbfgs",
                                           class_weight="balanced" if cfg["lr_balanced"] else None,
                                           random_state=seed).fit(X, yy)
    return xm, lm

def proba_ml(X, xm, lm):
    px = np.column_stack([xm[e].predict_proba(X)[:, 1] for e in SEMEVAL_LABELS]) if xm is not None else None
    pl = np.column_stack([lm[e].predict_proba(X)[:, 1] for e in SEMEVAL_LABELS]) if lm is not None else None
    return px, pl

def combine(px, pl, w):
    if px is None: return pl
    if pl is None: return px
    return w * px + (1 - w) * pl

def tune_w_ml(px, pl, y):
    best = (0.5, -np.inf)
    for w in WEIGHT_GRID:
        s = f1_score(y, (combine(px, pl, w) >= 0.5).astype(int), average="macro", zero_division=0)
        if s > best[1]: best = (float(w), s)
    return best[0]

def tune_th(p, y):
    out = []
    for j in range(p.shape[1]):
        base = float(y[:, j].mean()); cap = min(1.0, max(MAX_PPR_RATIO * base, base + 0.05)); sc = []
        for t in THRESHOLD_GRID:
            yp = (p[:, j] >= t).astype(int)
            if yp.mean() > cap: sc.append(-np.inf); continue
            prec = y[:, j][yp == 1].mean() if yp.sum() else 0
            if prec < MIN_PRECISION: sc.append(-np.inf); continue
            sc.append(f1_score(y[:, j], yp, zero_division=0))
        a = np.asarray(sc)
        if not np.isfinite(a).any():
            a = np.asarray([f1_score(y[:, j], (p[:, j] >= t).astype(int), zero_division=0) for t in THRESHOLD_GRID])
        out.append(float(THRESHOLD_GRID[np.where(a >= np.nanmax(a) - TIE_TOL)[0][-1]]))
    return np.asarray(out)

def oof_ml(X, y, cfg, seed):
    px = np.zeros(y.shape) if cfg["use_xgb"] else None; pl = np.zeros(y.shape) if cfg["use_lr"] else None
    for a, b in MultilabelStratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), y):
        tf, _, _ = fit_preprocessor(X[a], seed); xm, lm = train_ml(tf(X[a]), y[a], cfg, seed)
        qx, ql = proba_ml(tf(X[b]), xm, lm)
        if px is not None: px[b] = qx
        if pl is not None: pl[b] = ql
    return px, pl

# ============================================================
# 6. SINGLE-LABEL ENGINE (Tokopedia)
# ============================================================
def train_sl(X, y, cfg, seed):
    K = len(TOKO_CLASSES); xm = lm = None
    if cfg["use_xgb"]:
        sw = None
        if cfg["cost_sensitive"]:
            cnt = np.bincount(y, minlength=K); wt = len(y) / (K * np.maximum(cnt, 1)); sw = wt[y]
        xm = xgb.XGBClassifier(num_class=K, random_state=seed, **XGB_PARAMS_MULTI).fit(X, y, sample_weight=sw)
    if cfg["use_lr"]:
        lm = LogisticRegression(C=cfg["lr_C"], max_iter=1000, solver="lbfgs",
                                class_weight="balanced" if cfg["lr_balanced"] else None, random_state=seed).fit(X, y)
    return xm, lm

def proba_sl(X, xm, lm):
    return (xm.predict_proba(X) if xm is not None else None), (lm.predict_proba(X) if lm is not None else None)

def oof_sl(X, y, cfg, seed):
    K = len(TOKO_CLASSES)
    px = np.zeros((len(y), K)) if cfg["use_xgb"] else None; pl = np.zeros((len(y), K)) if cfg["use_lr"] else None
    for a, b in StratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), y):
        tf, _, _ = fit_preprocessor(X[a], seed); xm, lm = train_sl(tf(X[a]), y[a], cfg, seed)
        qx, ql = proba_sl(tf(X[b]), xm, lm)
        if px is not None: px[b] = qx
        if pl is not None: pl[b] = ql
    return px, pl

def tune_w_sl(px, pl, y):
    best = (0.5, -np.inf)
    for w in WEIGHT_GRID:
        s = f1_score(y, np.argmax(combine(px, pl, w), 1), average="macro", zero_division=0)
        if s > best[1]: best = (float(w), s)
    return best[0]

def tune_alpha(p, y):
    alpha = np.ones(len(TOKO_CLASSES))
    for _ in range(ALPHA_PASSES):
        for c in range(len(alpha)):
            sc = []
            for a in ALPHA_GRID:
                q = alpha.copy(); q[c] = a
                sc.append(f1_score(y, np.argmax(p * q, 1), average="macro", zero_division=0))
            sc = np.asarray(sc); cand = np.where(sc >= sc.max() - TIE_TOL)[0]
            alpha[c] = ALPHA_GRID[cand[np.argmin(np.abs(ALPHA_GRID[cand] - 1))]]
    return alpha

## 6. Model configurations and runners

In [ ]:
# ============================================================
# 7. CONFIGURATIONS
# ============================================================
def C(name, encoder, use_xgb=True, use_lr=True, cost_sensitive=False, lr_balanced=False, lr_C=1.0):
    return dict(name=name, encoder=encoder, use_xgb=use_xgb, use_lr=use_lr,
                cost_sensitive=cost_sensitive, lr_balanced=lr_balanced, lr_C=lr_C)

IMBALANCE_AWARE = dict(cost_sensitive=True, lr_balanced=True, lr_C=0.1)
ME_XBAL      = C("ME-XBAL", MAIN_ENCODER, **IMBALANCE_AWARE)          # proposed (E5)
ME_XBAL_XLMR = C("ME-XBAL (XLM-R)", "xlmr", **IMBALANCE_AWARE)        # second instantiation
CONFIGS = [
    C("XLM-R LR", "xlmr", use_xgb=False), C("XLM-R XGB", "xlmr", use_lr=False), C("XLM-R Hybrid", "xlmr"),
    C("IndoBERT LR", "indobert", use_xgb=False), C("IndoBERT XGB", "indobert", use_lr=False), C("IndoBERT Hybrid", "indobert"),
    C("E5 LR", "e5", use_xgb=False), C("E5 XGB", "e5", use_lr=False), C("E5 Hybrid", "e5"),
    ME_XBAL_XLMR, ME_XBAL,
]
print([c["name"] for c in CONFIGS])

# ============================================================
# 8. RUNNERS (return metrics + full artefacts for reproducibility)
# ============================================================
def run_sem(cfg, seed):
    E = SEMEVAL_EMB[cfg["encoder"]]
    dev = {l: np.r_[SEMEVAL_SPLITS[(l, seed)][0], SEMEVAL_SPLITS[(l, seed)][1]] for l in SEMEVAL_DATA}
    X = np.vstack([E[l][dev[l]] for l in SEMEVAL_DATA])
    y = np.vstack([SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int)[dev[l]] for l in SEMEVAL_DATA])
    px, pl = oof_ml(X, y, cfg, seed)
    w = tune_w_ml(px, pl, y) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    th = tune_th(combine(px, pl, w), y)
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_ml(tf(X), y, cfg, seed)
    rows, art = [], {"weight_xgb": w, "thresholds": th}
    for l in SEMEVAL_DATA:
        te = SEMEVAL_SPLITS[(l, seed)][2]
        yt = SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int)[te]
        pt = combine(*proba_ml(tf(E[l][te]), xm, lm), w); yp = (pt >= th[None, :]).astype(int)
        rows.append({"model": cfg["name"], "seed": seed, "lang": l, "n_test": len(te), "weight_xgb": w,
                     "macro_f1": f1_score(yt, yp, average="macro", zero_division=0)})
        art[l] = {"test_idx": te, "y_true": yt, "y_pred": yp, "proba": pt}
    return rows, art

def run_tok(cfg, seed, train_idx=None, val_idx=None):
    E = TOKO_EMB[cfg["encoder"]]; tr, va, te = TOKO_SPLITS[seed]
    if train_idx is not None: tr = train_idx
    if val_idx is not None: va = val_idx
    dev = np.r_[tr, va]; X = E[dev]; y = TOKO_DF.label.values[dev]
    px, pl = oof_sl(X, y, cfg, seed)
    w = tune_w_sl(px, pl, y) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    alpha = tune_alpha(combine(px, pl, w), y)
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_sl(tf(X), y, cfg, seed)
    pt = combine(*proba_sl(tf(E[te]), xm, lm), w); yt = TOKO_DF.label.values[te]; yp = np.argmax(pt * alpha, 1)
    row = {"model": cfg["name"], "seed": seed, "n_test": len(te), "weight_xgb": w,
           "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
           "micro_f1": f1_score(yt, yp, average="micro", zero_division=0), "accuracy": accuracy_score(yt, yp)}
    art = {"weight_xgb": w, "alpha": alpha, "test_idx": te, "y_true": yt, "y_pred": yp, "proba": pt}
    return row, art

## 7. E1/E3 — main comparison over ten seeds (SemEval and Tokopedia)

In [ ]:
# ============================================================
# 9. MAIN EXPERIMENT (resumable)
# ============================================================
SEM_RAW, TOK_RAW = f"{OUT_ROOT}/SemEval_final_raw.csv", f"{OUT_ROOT}/Tokopedia_final_raw.csv"
if RUN_MAIN:
    for seed in SEEDS:
        sem_done = done_keys(SEM_RAW, ["model", "seed"]); tok_done = done_keys(TOK_RAW, ["model", "seed"])
        for cfg in tqdm(CONFIGS, desc=f"seed {seed}"):
            key = (cfg["name"], str(seed)); safe = re.sub(r"[^A-Za-z0-9]+", "_", cfg["name"])
            if key not in sem_done:
                rows, art = run_sem(cfg, seed); append_csv(SEM_RAW, rows)
                joblib.dump(art, f"{OUT_ROOT}/predictions/SemEval_{safe}_seed{seed}.joblib", compress=3)
            if key not in tok_done:
                row, art = run_tok(cfg, seed); append_csv(TOK_RAW, [row])
                joblib.dump(art, f"{OUT_ROOT}/predictions/Tokopedia_{safe}_seed{seed}.joblib", compress=3)
df_sem, df_tok = pd.read_csv(SEM_RAW), pd.read_csv(TOK_RAW)
print(df_sem.groupby(["model", "lang"]).macro_f1.mean().unstack().round(4))
print(df_tok.groupby("model")[["macro_f1", "micro_f1"]].mean().round(4))

## 8. Statistics and manuscript tables

In [ ]:
# ============================================================
# 10. STATISTICS (corrected resampled t-test, Wilcoxon, Holm)
# ============================================================
ORDER = [c["name"] for c in CONFIGS]
REF = "ME-XBAL"
RATIO_SEM = 0.20 / 0.80                         # n_test / n_dev
RATIO_TOK = len(TOKO_SPLITS[SEED_PRIMARY][2]) / (len(TOKO_SPLITS[SEED_PRIMARY][0]) + len(TOKO_SPLITS[SEED_PRIMARY][1]))
ms = lambda v: f"{np.mean(v):.4f} ± {np.std(v, ddof=1):.4f}"

def compare(piv, ref, ratio, models):
    rows = []
    for m in models:
        if m == ref: continue
        d = (piv[ref] - piv[m]).values; mm, lo, hi, p = corrected_ttest(d, ratio)
        rows.append(dict(comparison=f"{ref} vs {m}", delta=mm, ci95_low=lo, ci95_high=hi, p_corrected=p,
                         p_wilcoxon=wilcoxon_p(piv[ref], piv[m]), wins=int((d > 0).sum()), n_seeds=len(d)))
    out = pd.DataFrame(rows); out["p_holm"] = holm(out.p_corrected); return out

sem_avg = df_sem.pivot_table(index="seed", columns="model", values="macro_f1", aggfunc="mean")
tok_piv = df_tok.pivot(index="seed", columns="model", values="macro_f1")

# Table: SemEval per language (mean ± SD)
t_sem = pd.DataFrame([{"model": m, **{l: ms(df_sem[(df_sem.model == m) & (df_sem.lang == l)].macro_f1) for l in ["sun", "jav", "ind"]},
                       "average": ms(sem_avg[m])} for m in ORDER])
t_sem_cmp = compare(sem_avg, REF, RATIO_SEM, ORDER)
t_sem_lang = pd.concat([compare(df_sem[df_sem.lang == l].pivot(index="seed", columns="model", values="macro_f1"), REF, RATIO_SEM, ORDER)
                        .assign(lang=l) for l in ["sun", "jav", "ind"]])
t_tok = pd.DataFrame([{"model": m, "macro_f1": ms(tok_piv[m]), "micro_f1": df_tok[df_tok.model == m].micro_f1.mean()} for m in ORDER])
t_tok_cmp = compare(tok_piv, REF, RATIO_TOK, ORDER)

# One-factor contrasts
CONTRASTS = [("Imbalance-aware fitting, E5 (ME-XBAL − E5 Hybrid)", "ME-XBAL", "E5 Hybrid"),
             ("Imbalance-aware fitting, XLM-R (ME-XBAL (XLM-R) − XLM-R Hybrid)", "ME-XBAL (XLM-R)", "XLM-R Hybrid"),
             ("Boosting branch, E5 (E5 Hybrid − E5 LR)", "E5 Hybrid", "E5 LR"),
             ("Boosting branch, XLM-R (XLM-R Hybrid − XLM-R LR)", "XLM-R Hybrid", "XLM-R LR"),
             ("LR branch, E5 (E5 Hybrid − E5 XGB)", "E5 Hybrid", "E5 XGB"),
             ("Encoder, E5 vs. XLM-R (Hybrid)", "E5 Hybrid", "XLM-R Hybrid"),
             ("Encoder, E5 vs. XLM-R (imbalance-aware head)", "ME-XBAL", "ME-XBAL (XLM-R)"),
             ("Encoder, XLM-R vs. IndoBERT (Hybrid)", "XLM-R Hybrid", "IndoBERT Hybrid")]
rows = []
for lab, a, b in CONTRASTS:
    r = {"contrast": lab}
    for nm, piv, ratio in [("semeval", sem_avg, RATIO_SEM), ("tokopedia", tok_piv, RATIO_TOK)]:
        mm, lo, hi, p = corrected_ttest(piv[a] - piv[b], ratio)
        r.update({f"{nm}_delta": mm, f"{nm}_ci_low": lo, f"{nm}_ci_high": hi, f"{nm}_p": p,
                  f"{nm}_wins": int(((piv[a] - piv[b]) > 0).sum())})
    rows.append(r)
t_contrast = pd.DataFrame(rows)

# Selected XGBoost weights
t_weights = pd.concat([df_sem.groupby("model").weight_xgb.mean().rename("semeval"),
                       df_tok.groupby("model").weight_xgb.mean().rename("tokopedia")], axis=1)

for name, t in [("table_semeval", t_sem), ("table_semeval_comparison", t_sem_cmp), ("table_semeval_per_language", t_sem_lang),
                ("table_tokopedia", t_tok), ("table_tokopedia_comparison", t_tok_cmp), ("table_contrasts", t_contrast),
                ("table_selected_weights", t_weights.reset_index())]:
    t.to_csv(f"{OUT_ROOT}/tables/{name}.csv", index=False)
pd.set_option("display.width", 250)
print(t_sem.to_string(index=False)); print(t_sem_cmp.round(4).to_string(index=False))
print(t_tok.to_string(index=False)); print(t_tok_cmp.round(4).to_string(index=False))
print(t_contrast.round(4).to_string(index=False))

In [ ]:
# ============================================================
# 11. PRIMARY-SEED PAIRED PREDICTION BOOTSTRAP
# ============================================================
def load_art(ds, name, seed):
    return joblib.load(f"{OUT_ROOT}/predictions/{ds}_{re.sub(r'[^A-Za-z0-9]+', '_', name)}_seed{seed}.joblib")

ref_tok = load_art("Tokopedia", REF, SEED_PRIMARY); ref_sem = load_art("SemEval", REF, SEED_PRIMARY)
rows = []
for m in ORDER:
    if m == REF: continue
    a = load_art("Tokopedia", m, SEED_PRIMARY)
    rows.append({"dataset": "Tokopedia", "lang": "ind", "comparison": f"{REF} - {m}",
                 **paired_prediction_bootstrap(ref_tok["y_true"], ref_tok["y_pred"], a["y_pred"])})
    a = load_art("SemEval", m, SEED_PRIMARY)
    for l in SEMEVAL_DATA:
        rows.append({"dataset": "SemEval", "lang": l, "comparison": f"{REF} - {m}",
                     **paired_prediction_bootstrap(ref_sem[l]["y_true"], ref_sem[l]["y_pred"], a[l]["y_pred"])})
t_boot = pd.DataFrame(rows); t_boot.to_csv(f"{OUT_ROOT}/tables/table_primary_seed_bootstrap.csv", index=False)
print(t_boot.round(4).to_string(index=False))

## Ablation C: E5 Hybrid with C = 0.1, unweighted

In [ ]:
# ============================================================
# ABLATION C (B2): isolates class weighting from LR regularization
# ============================================================
ABL = C("E5 Hybrid (C = 0.1)", "e5", lr_C=0.1)          # no class weights, C = 0.1
ABL_SEM, ABL_TOK = f"{OUT_ROOT}/SemEval_ablationC_raw.csv", f"{OUT_ROOT}/Tokopedia_ablationC_raw.csv"
for seed in SEEDS:
    if (ABL["name"], str(seed)) not in done_keys(ABL_SEM, ["model", "seed"]):
        rows, _ = run_sem(ABL, seed); append_csv(ABL_SEM, rows)
    if (ABL["name"], str(seed)) not in done_keys(ABL_TOK, ["model", "seed"]):
        row, _ = run_tok(ABL, seed); append_csv(ABL_TOK, [row])
a_sem = pd.concat([df_sem, pd.read_csv(ABL_SEM)]).pivot_table(index="seed", columns="model", values="macro_f1", aggfunc="mean")
a_tok = pd.concat([df_tok, pd.read_csv(ABL_TOK)]).pivot(index="seed", columns="model", values="macro_f1")
rows = []
for lab, a, b in [("Class weighting (ME-XBAL − E5 Hybrid C=0.1)", "ME-XBAL", "E5 Hybrid (C = 0.1)"),
                  ("LR regularization (E5 Hybrid C=0.1 − E5 Hybrid)", "E5 Hybrid (C = 0.1)", "E5 Hybrid"),
                  ("Whole head (ME-XBAL − E5 Hybrid)", "ME-XBAL", "E5 Hybrid")]:
    r = {"contrast": lab}
    for nm, piv, ratio in [("semeval", a_sem, RATIO_SEM), ("tokopedia", a_tok, RATIO_TOK)]:
        mm, lo, hi, p = corrected_ttest(piv[a] - piv[b], ratio)
        r.update({f"{nm}_delta": mm, f"{nm}_ci_low": lo, f"{nm}_ci_high": hi, f"{nm}_p": p, f"{nm}_wins": int(((piv[a] - piv[b]) > 0).sum())})
    rows.append(r)
t_abl = pd.DataFrame(rows); t_abl.to_csv(f"{OUT_ROOT}/tables/table_ablationC.csv", index=False)
print(a_sem[["ME-XBAL", "E5 Hybrid", "E5 Hybrid (C = 0.1)"]].mean().round(4)); print(a_tok[["ME-XBAL", "E5 Hybrid", "E5 Hybrid (C = 0.1)"]].mean().round(4))
print(t_abl.round(4).to_string(index=False))
shutil.make_archive(f"{WORK}/ME-XBAL_ablationC_outputs", "zip", OUT_ROOT)